**Assignment 4 - Unsupervised Learning Clustering and Recommendations**

**Information about the group and report**

Group Number: 58 \\
Name of Team Members: Yasmine Zoubdi and Anoushka Jawale \\
Student Numbers : 300170464 and 300233148 \\

In [ ]:
!pip install python-Levenshtein

import pandas as pd
import numpy as np  # Only one numpy import
from sklearn.metrics import jaccard_score, mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances, manhattan_distances
from scipy.spatial.distance import euclidean, cityblock, hamming
import matplotlib.pyplot as plt
import seaborn as sns
from Levenshtein import distance as levenshtein_distance
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler, LabelEncoder

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 19.5 MB/s eta 0:00:00


###**First look at data:**

 **Amazon-clothing-info.csv**

In [ ]:
url = "https://raw.githubusercontent.com/Yasmine08POG/CSI4142/refs/heads/main/Amazon-clothing-info.csv"
clothing_info = pd.read_csv(url)
clothing_info.head()

,sku,asin,product_type_name,formatted_price,author,color,brand,publisher,availability,reviews,large_image_url,availability_type,small_image_url,editorial_review,title,model,medium_image_url,manufacturer,editorial_reivew
0,NaN,B004GSI2OS,SHIRT,$26.26,NaN,Onyx Black/ Stone,FeatherLite,NaN,Usually ships in 6-10 business days,"[False, 'https://www.amazon.com/reviews/iframe...",https://images-na.ssl-images-amazon.com/images...,now,https://images-na.ssl-images-amazon.com/images...,NaN,Featherlite Ladies' Long Sleeve Stain Resistan...,NaN,https://images-na.ssl-images-amazon.com/images...,NaN,NaN
1,NaN,B012YX2ZPI,SHIRT,$9.99,NaN,White,HX-Kingdom Fashion T-shirts,NaN,Usually ships in 4-5 business days,"[False, 'https://www.amazon.com/reviews/iframe...",https://images-na.ssl-images-amazon.com/images...,now,https://images-na.ssl-images-amazon.com/images...,This Personalized Special Olympics World Games...,Women's Unique 100% Cotton T - Special Olympic...,NaN,https://images-na.ssl-images-amazon.com/images...,NaN,NaN
2,NaN,B001LOUGE4,SHIRT,$11.99,NaN,Black,Fitness Etc.,NaN,NaN,"[False, 'https://www.amazon.com/reviews/iframe...",https://images-na.ssl-images-amazon.com/images...,NaN,https://images-na.ssl-images-amazon.com/images...,Light Weight 2x1 Boy Beater Tank Top. Great t...,Ladies Cotton Tank 2x1 Ribbed Tank Top,NaN,https://images-na.ssl-images-amazon.com/images...,NaN,NaN
3,HT-2001_Lime-1149-XL,B003BSRPB0,SHIRT,$20.54,NaN,White,FeatherLite,NaN,Usually ships in 6-10 business days,"[False, 'https://www.amazon.com/reviews/iframe...",https://images-na.ssl-images-amazon.com/images...,now,https://images-na.ssl-images-amazon.com/images...,FeatherLite Ladies' Moisture Free Mesh Sport S...,FeatherLite Ladies' Moisture Free Mesh Sport S...,NaN,https://images-na.ssl-images-amazon.com/images...,NaN,NaN
4,NaN,B014ICEDNA,SHIRT,$7.50,NaN,Purple,FNC7C,NaN,Usually ships in 4-5 business days,"[True, 'https://www.amazon.com/reviews/iframe?...",https://images-na.ssl-images-amazon.com/images...,now,https://images-na.ssl-images-amazon.com/images...,Supernatural Chibis Sam Dean And Castiel Women...,Supernatural Chibis Sam Dean And Castiel Short...,NaN,https://images-na.ssl-images-amazon.com/images...,NaN,NaN


**Clothing-Reviews.csv**

In [ ]:
url = "https://raw.githubusercontent.com/Yasmine08POG/CSI4142/refs/heads/main/Clothing-Reviews.csv"
reviews = pd.read_csv(url)
reviews.head()

,asin,title,review_userId,review_score,review_summary,review_text
0,B004GSI2OS,Featherlite Ladies' Long Sleeve Stain Resistan...,A174NPQZ1EABX1,5,"Fundamental of death metal, do not miss","A roaring onslaught of streaming sound, this a..."
1,B004GSI2OS,Featherlite Ladies' Long Sleeve Stain Resistan...,A17ZI8VKZJFOV8,5,Great Classic,One of incantations best recordings. Im a fan ...
2,B004GSI2OS,Featherlite Ladies' Long Sleeve Stain Resistan...,A2F9LXG1QEJ855,4,Very Original Debut of one of the most striden...,"This is an early, and rough-sounding look at t..."
3,B004GSI2OS,Featherlite Ladies' Long Sleeve Stain Resistan...,A2FGXHSHF0OD17,5,No secret here,Whats to say about this album? Quintessential ...
4,B004GSI2OS,Featherlite Ladies' Long Sleeve Stain Resistan...,A2O2E8BLB7VW1U,5,Brings a smile to my face...,...Whenever i listing to an album by incantati...


**a) Description of the Dataset**

**Dataset Name:** Amazon Apparels Data

**Author:** thekenjin

**Purpose:**\
The dataset was assembled to capture detailed information about apparel products listed on Amazon. It is designed to support studies in exploratory data analysis, pricing strategies, customer review analysis, and e-commerce trends within the apparel industry. Researchers can use this data to build predictive models (e.g., recommendation systems) and to gain insights into market dynamics. The data is divided into two datsets:

* Amazon-clothing-info.csv **containing metadata** about clothing.
* Clothing-Reviews.csv gives a review_score and a review_userId which we will need for the collaborative filtering.


**b) Data Dictionary**

The dataset **Amazon-clothing-info.csv** contains **28,395 product records** with **19 features**, while **Clothing-Reviews.csv** includes **50,046 product reviews** and **6 features**. Below is a data dictionary describing the features in both datasets.

######Amazon-clothing-info.csv

| **Feature**           | **Data Type**                | **Description**                                                                                                                                                                             |
|---------------------|------------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| sku                 | Categorical                  | Unique stock-keeping unit identifier for the product.                                                                                                          |
| asin                | Categorical                  | Amazon Standard Identification Number, serving as the unique key to link product metadata with reviews.                                                                                     |
| product_type_name   | Categorical                  | Specifies the type of apparel product (e.g., SHIRT).                                                                                                                                       |
| formatted_price     | Numerical (stored as string) | The product’s price formatted as a currency string (e.g., "$26.26"). For analysis, convert this field to a numerical value by stripping currency symbols.                              |
| author              | Categorical                  | May indicate the designer or contributor of the product information.                                                                                                                       |
| color               | Categorical                  | The color of the product (e.g., "Onyx Black/ Stone").                                                                                                                                        |
| brand               | Categorical                  | The brand or label under which the product is marketed (e.g., FeatherLite).                                                                                                                  |
| publisher           | Categorical                  | Typically indicates the seller or source information; included as part of the product metadata.                                                                                            |
| availability        | Categorical                  | Describes the shipping status (e.g., "Usually ships in 6-10 business days").                                                                                                                 |
| reviews             | Complex/Categorical          | Contains review-related metadata (e.g., a flag and URL) but does not include detailed review scores; detailed reviews are available in the separate reviews file.                         |
| large_image_url     | Categorical                  | URL linking to a large version of the product image.                                                                                                                                       |
| availability_type   | Categorical                  | Specifies the type or category of availability (e.g., "now").                                                                                                                              |
| small_image_url     | Categorical                  | URL linking to a small version of the product image.                                                                                                                                       |
| editorial_review    | Categorical/Text             | Contains expert or editorial commentary on the product.                                                                                                                                    |
| title               | Categorical                  | Full title or name of the product as listed on Amazon.                                                                                                                                      |
| model               | Categorical                  | The model identifier for the apparel item, useful for differentiating product variations.                                                                                                  |
| medium_image_url    | Categorical                  | URL for a medium-sized version of the product image.                                                                                                                                       |
| manufacturer        | Categorical                  | Indicates the company that manufactured the product.                                                                                                                                       |
| editorial_reivew    | Categorical/Text             | Likely a duplicate or alternative version of “editorial_review,” possibly containing additional commentary.                                                                                  |

######Clothing-Reviews.csv

| **Field**       | **Data Type**    | **Description**                                                                                                                                                                    |
|-----------------|------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| asin            | Categorical      | Amazon Standard Identification Number used to link each review to its corresponding product metadata in the Amazon-clothing-info.csv file.                                          |
| title           | Categorical      | The product title as recorded in the review data, which may duplicate metadata but helps in text-based filtering or cross-verification.                                             |
| review_userId   | Categorical      | Unique identifier for the user who submitted the review, essential for user-based collaborative filtering.                                                                       |
| review_score    | Numerical        | The score assigned in the review, used for calculating overall product ratings and for collaborative filtering.                                                                  |
| review_summary  | Categorical/Text | A brief summary of the review that provides insight into the user’s sentiment.                                                                                                    |
| review_text     | Categorical/Text | The full text of the review, offering detailed feedback that can be utilized for sentiment analysis or qualitative assessment.                                                      |

### **Data preprocessing :**

##### **Duplicate Columns:**

Before beginning our analysis, we identified duplicate columns for the feature `editorial_review`, one of which is misspelled as `editorial_reivew`. We will investigate and resolve this discrepancy during the data preprocessing phase to ensure data integrity.  

We suspect that these columns are complimentary but to confirm this hypothesis we conduct the following test:

In [ ]:
# 1. Compare missing counts in both columns
missing_counts = clothing_info[['editorial_review', 'editorial_reivew']].isnull().sum()
print("Missing counts:")
print(missing_counts)

# 2. Filter rows where both columns are not missing
clothing_info_both = clothing_info[clothing_info['editorial_review'].notna() & clothing_info['editorial_reivew'].notna()]
print("\nNumber of rows with both columns non-missing:", len(clothing_info_both))

# 3. Analyze missing patterns:
# Count rows where one column is missing and the other is present
missing_review = clothing_info[clothing_info['editorial_review'].isnull() & clothing_info['editorial_reivew'].notna()]
missing_reivew = clothing_info[clothing_info['editorial_reivew'].isnull() & clothing_info['editorial_review'].notna()]
print("\nRows where 'editorial_review' is missing but 'editorial_reivew' is available:", len(missing_review))
print("Rows where 'editorial_reivew' is missing but 'editorial_review' is available:", len(missing_reivew))


Missing counts:
editorial_review    27954
editorial_reivew     2841
dtype: int64

Number of rows with both columns non-missing: 0

Rows where 'editorial_review' is missing but 'editorial_reivew' is available: 25554
Rows where 'editorial_reivew' is missing but 'editorial_review' is available: 441


Instead of discarding the column `editorial_review` we decide to impute the missing values in `editorial_reivew`with the present values in `editorial_review`.

In [ ]:
# Impute missing values in 'editorial_reivew' using values from 'editorial_review'
clothing_info['editorial_reivew'] = clothing_info['editorial_reivew'].fillna(clothing_info['editorial_review'])

# Optional: Check the missing count after imputation
print("Missing values in 'editorial_reivew' after imputation:", clothing_info['editorial_reivew'].isnull().sum())

Missing values in 'editorial_reivew' after imputation: 2400


##### **Missing data:**

In [ ]:
missing_table = pd.DataFrame({
    'Missing Values': clothing_info.isnull().sum(),
    'Missing Percentage': (clothing_info.isnull().sum() / len(clothing_info)) * 100
})
pd.set_option('display.max_rows', None)
missing_table

,Missing Values,Missing Percentage
sku,28261,99.528086
asin,0,0.000000
product_type_name,0,0.000000
formatted_price,0,0.000000
author,28394,99.996478
color,10,0.035217
brand,93,0.327522
publisher,20054,70.625110
availability,3863,13.604508
reviews,0,0.000000


**Variables removal:** \
 The percentage of missing data for the columns sku, author, publisher, model, editorial_review and manufacturer is above 70% so we decided to delete these columns.

In [ ]:
columns_to_drop = missing_table[missing_table['Missing Percentage'] > 70].index
clothing_info = clothing_info.drop(columns=columns_to_drop)

**listwise deletion :** \
Since the percentage of missing values for the variables left is less than 14%. We decided to simply remove rows with missing data.

In [ ]:
clothing_info = clothing_info.dropna()

##### **Data Type errors:**

In this step, we noticed that the `formatted_price` is not of the right data type. The values for this column should be numerical but instead it is a string value.

In [ ]:
clothing_info['price'] = pd.to_numeric(clothing_info['formatted_price'].str.replace('$', '', regex=False), errors='coerce')
clothing_info.drop(columns=['formatted_price'], inplace=True)
clothing_info.dropna(subset=['price'], inplace=True) # Only 0.09901% NA values

<ipython-input-9-da4195b911ea>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clothing_info['price'] = pd.to_numeric(clothing_info['formatted_price'].str.replace('$', '', regex=False), errors='coerce')


##### **Data transformation:**

We noticed that the `color` feature is of the format "Color1/ Color2". For the purpose of our study we will perform a splitting to break down the composite string in the `color` column into individual components, resulting in a set of distinct color values like {"Onyx Black", "Stone"}.

In [ ]:
# Split colors into sets
clothing_info['color'] = clothing_info['color'].str.split('/').apply(
    lambda x: set([c.strip() for c in x]) if isinstance(x, list) else set()
)

print(clothing_info['color'].head())

1     {White}
3     {White}
4    {Purple}
5    {Purple}
6     {White}
Name: color, dtype: object


We can do the same for the `brand` feature.

In [ ]:
# Split brands into sets (assuming '/' is the delimiter)
clothing_info['brand'] = (
    clothing_info['brand']
    .str.split('/')  # Split by delimiter
    .apply(
        lambda x: set([b.strip() for b in x])
        if isinstance(x, list)  # Check if split result is a list
        else set()  # Handle NaN or invalid entries
    )
)


### **Study 1: Similarity measures**

In [ ]:
print(clothing_info.head())

         asin product_type_name     color                          brand  \
1  B012YX2ZPI             SHIRT   {White}  {HX-Kingdom Fashion T-shirts}   
3  B003BSRPB0             SHIRT   {White}                  {FeatherLite}   
4  B014ICEDNA             SHIRT  {Purple}                        {FNC7C}   
5  B014ICEJ1Q             SHIRT  {Purple}                        {FNC7C}   
6  B0079BMKDS           APPAREL   {White}                  {FeatherLite}   

                          availability  \
1   Usually ships in 4-5 business days   
3  Usually ships in 6-10 business days   
4   Usually ships in 4-5 business days   
5   Usually ships in 4-5 business days   
6   Usually ships in 2-3 business days   

                                             reviews  \
1  [False, 'https://www.amazon.com/reviews/iframe...   
3  [False, 'https://www.amazon.com/reviews/iframe...   
4  [True, 'https://www.amazon.com/reviews/iframe?...   
5  [False, 'https://www.amazon.com/reviews/iframe...   
6  [False,

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import euclidean_distances, manhattan_distances
from sklearn.preprocessing import LabelEncoder
from scipy.spatial.distance import hamming
from Levenshtein import distance as levenshtein_distance

def jaccard_similarity(set1, set2):
    if not isinstance(set1, set) or not isinstance(set2, set):
        return 0  # Return 0 similarity for invalid inputs
    return len(set1 & set2) / len(set1 | set2) if len(set1 | set2) > 0 else 0


def tfidf_vectorizer_similarity(text1, text2):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([text1, text2])
    return np.dot(tfidf_matrix[0].toarray(), tfidf_matrix[1].toarray().T)[0][0]

def compute_euclidean(value1, value2):
    return euclidean_distances([[value1]], [[value2]])[0][0]

def compute_hamming(set1, set2):
    if not isinstance(set1, set) or not isinstance(set2, set):
        return None  # Hamming distance requires sets
    if len(set1) != len(set2):
        return None  # Hamming distance requires equal lengths
    return 1 - hamming(list(set1), list(set2))

def find_similar_products(df, query, attribute, similarity_func, top_n=10):
    similarities = df[attribute].apply(lambda x: similarity_func(query, x) if pd.notnull(x) else 0)
    df['similarity_score'] = similarities
    return df.sort_values(by='similarity_score', ascending=False).head(top_n)

def simulate_requests(df):
    requests = [
        (set(["Purple"]), "color", jaccard_similarity),
        (26.26, "price", lambda q, x: -abs(q - x)),
        ("Classic Fit Polo Shirt", "title", lambda q, x: -levenshtein_distance(q, x)),
        (set(["FeatherLite"]), "brand", compute_hamming),
        (set(["now"]), "availability_type", jaccard_similarity)
    ]

    for query, attribute, similarity_func in requests:
        print(f"\nTop 10 similar products for {attribute} matching '{query}':")
        top_products = find_similar_products(df, query, attribute, similarity_func)
        print(top_products[["asin", "title", attribute, "similarity_score"]])

# Sample usage:
simulate_requests(clothing_info)



Top 10 similar products for color matching '{'Purple'}':
            asin                                              title     color  \
8825  B06XK9V13R  Paradox Women's DriRelease 1/4 Zip Top Base La...  {Purple}   
8832  B06ZZT2NLK  Handkerchief Tunic Top Long Sleeve Passion Ber...  {Purple}   
8841  B071KRVDGM  Gilligan & O'Malley Womens Halter Top Sleep Sk...  {Purple}   
8896  B01MR4QDEV  Miraclebody by Miraclesuit Womens Shaping Scar...  {Purple}   
9005  B01F2UDMXE  Southern Lady Women's Scoop Neck W/Knot Detail...  {Purple}   
9037  B06VW4H2RT  Zanzea Womens Long Sleeve Lace Crochet Crop To...  {Purple}   
9050  B074LC5TPL  KpopBaby Womens Long Sleeve Knitted Casual Car...  {Purple}   
9077  B01LZ8PFY3  KingYuan 1 Pair Women Fetish Bondage Breast Ni...  {Purple}   
9105  B074P8VTBG  Becca By Rebecca Virtue Magenta Women's Solid ...  {Purple}   
490   B01CH48FVC  Fifteen-Twenty Women's Silk Ruffle Blouse, Pur...  {Purple}   

      similarity_score  
8825               1.0  


### **Study 2: Clustering algorithms**

In [ ]:
# Load data
df = clothing_info.copy()

# Ensure that categorical columns are correctly handled before encoding
df['brand_encoded'] = LabelEncoder().fit_transform(df['brand'].astype(str))
df['product_type_name_encoded'] = LabelEncoder().fit_transform(df['product_type_name'].astype(str))

# KMeans Clustering
def kmeans_clustering(df, k, attributes):
    kmeans = KMeans(n_clusters=k, random_state=42)
    X = df[attributes]
    X_scaled = StandardScaler().fit_transform(X)
    kmeans.fit(X_scaled)
    df['kmeans_cluster'] = kmeans.labels_
    return df

# DBSCAN Clustering
def dbscan_clustering(df, eps, min_samples, attributes):
    X = df[attributes]
    X_scaled = StandardScaler().fit_transform(X)
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    df['dbscan_cluster'] = dbscan.fit_predict(X_scaled)
    return df

# Visualize clustering results
def visualize_clusters(df, attribute1, attribute2, cluster_column, title):
    plt.figure(figsize=(8, 6))
    plt.scatter(df[attribute1], df[attribute2], c=df[cluster_column], cmap='viridis', s=50)
    plt.title(title)
    plt.xlabel(attribute1)
    plt.ylabel(attribute2)
    plt.colorbar(label='Cluster')
    plt.show()

# KMeans clustering with k=3 and k=5
df_kmeans_3 = kmeans_clustering(df, 3, ['price', 'color'])
df_kmeans_5 = kmeans_clustering(df, 5, ['price', 'color'])

# DBSCAN clustering with eps=0.5 and min_samples=5, eps=0.3 and min_samples=10
df_dbscan_1 = dbscan_clustering(df, 0.5, 5, ['brand_encoded', 'product_type_name_encoded'])
df_dbscan_2 = dbscan_clustering(df, 0.3, 10, ['brand_encoded', 'product_type_name_encoded'])

# Visualize results
visualize_clusters(df_kmeans_3, 'price', 'color', 'kmeans_cluster', 'KMeans Clustering (k=3)')
visualize_clusters(df_kmeans_5, 'price', 'color', 'kmeans_cluster', 'KMeans Clustering (k=5)')

visualize_clusters(df_dbscan_1, 'brand_encoded', 'product_type_name_encoded', 'dbscan_cluster', 'DBSCAN Clustering (eps=0.5, min_samples=5)')
visualize_clusters(df_dbscan_2, 'brand_encoded', 'product_type_name_encoded', 'dbscan_cluster', 'DBSCAN Clustering (eps=0.3, min_samples=10)')


NameError: name 'clothing_info' is not defined

### **Study 3: Content-Based Recommendation System**

#### **Heuristic Descriptions**
1. **Heuristic 1 (Product-Type + Color + Price)**  
   - **Features**:  
     - `product_type_name`: Jaccard similarity (categorical).  
     - `color`: Jaccard similarity (categorical).  
     - `price`: Euclidean distance (numerical, converted to similarity via `1 / (1 + distance)`).  
   - **Domain Justification**: Users often prefer items of the same type (e.g., shirts), specific colors, and within a price range. Combining these captures functional and aesthetic preferences.

2. **Heuristic 2 (Brand + Title + Price)**  
   - **Features**:  
     - `brand`: Jaccard similarity (categorical).  
     - `title`: TF-IDF cosine similarity (textual).  
     - `price`: Manhattan distance (numerical, converted to similarity via `1 / (1 + distance)`).  
   - **Domain Justification**: Brand loyalty and descriptive keywords in titles (e.g., "Moisture Free Mesh") influence preferences, alongside budget constraints.

#### **Code Implementation**

In [ ]:
# Feature Selection
clothing_info = clothing_info.dropna(subset=['product_type_name', 'color', 'brand', 'title', 'price'])
clothing_info = clothing_info.reset_index(drop=True)
clothing_info['price'] = clothing_info['price'].astype(float)

# TF-IDF Vectorizer for titles
tfidf = TfidfVectorizer(stop_words='english')
title_matrix = tfidf.fit_transform(clothing_info['title'])

# Function to compute Jaccard similarity between sets
def jaccard_similarity(set1, set2):
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union != 0 else 0

# Function to compute Heuristic 1 similarity
def heuristic1_similarity(target_idx, dataset):
    target = dataset.iloc[target_idx]
    similarities = []

    for idx, row in dataset.iterrows():
        # Jaccard for product_type (single-value)
        product_sim = 1 if target['product_type_name'] == row['product_type_name'] else 0

        # Jaccard for color (set-based)
        color_sim = jaccard_similarity(target['color'], row['color'])

        # Price similarity (Euclidean)
        price_sim = 1 / (1 + euclidean([target['price']], [row['price']]))

        # Combined similarity
        total_sim = (product_sim + color_sim + price_sim) / 3
        similarities.append(total_sim)

    return similarities

# Function to compute Heuristic 2 similarity
def heuristic2_similarity(target_idx, dataset, title_matrix):
    target = dataset.iloc[target_idx]
    similarities = []

    title_sim = cosine_similarity(title_matrix[target_idx], title_matrix).flatten()

    for pos_idx, (_, row) in enumerate(dataset.iterrows()):
        brand_sim = jaccard_similarity(target['brand'], row['brand'])
        price_sim = 1 / (1 + cityblock([target['price']], [row['price']]))
        total_sim = (brand_sim + title_sim[pos_idx] + price_sim) / 3
        similarities.append(total_sim)

    return similarities

# Compute similarities

# Example: Target item (FeatherLite Sport Shirt)
target_idx = clothing_info[clothing_info['title'].str.contains("FeatherLite Ladies' Moisture Free Mesh Sport Shirt")].index[0]

# Compute similarities
heuristic1_scores = heuristic1_similarity(target_idx, clothing_info)
heuristic2_scores = heuristic2_similarity(target_idx, clothing_info, title_matrix)

# Function to generate recommendations
def generate_recommendations(query_title, dataset, title_matrix):
    # Find target item
    target_idx = dataset[dataset['title'].str.contains(query_title, case=False)].index[0]

    # Compute similarities
    heuristic1_scores = heuristic1_similarity(target_idx, dataset)
    heuristic2_scores = heuristic2_similarity(target_idx, dataset, title_matrix)

    # Add scores to DataFrame
    dataset['heuristic1_rank'] = heuristic1_scores
    dataset['heuristic2_rank'] = heuristic2_scores

    # Get top 10 results for both heuristics
    top10_heuristic1 = dataset.sort_values('heuristic1_rank', ascending=False).head(10)[
        ['title', 'product_type_name', 'color', 'price']
    ]
    top10_heuristic2 = dataset.sort_values('heuristic2_rank', ascending=False).head(10)[
        ['title', 'brand', 'price']
    ]

    return top10_heuristic1, top10_heuristic2

# Simulate 3 requests
requests = [
    "Women's Unique 100% Cotton T - Special Olympics World Games 2015 White Size L",
    "FeatherLite Ladies' Moisture Free Mesh Sport Shirt, White , XXX-Large",
    "Supernatural Chibis Sam Dean And Castiel O Neck T-shirts For Female Purple L"
]

# Generate results
results = {}
for query in requests:
    results[query] = generate_recommendations(query, clothing_info, title_matrix)

#### **Results: Request 1**

1. **Heuristic 1 (Product-Type + Color + Price):**

In [ ]:
results["Women's Unique 100% Cotton T - Special Olympics World Games 2015 White Size L"][0]

2. **Heuristic 2 (Brand + Title + Price)**  

In [ ]:
results["Women's Unique 100% Cotton T - Special Olympics World Games 2015 White Size L"][1]

**Discussion of Results**

**Query**: `Women's Unique 100% Cotton T - Special Olympics World Games 2015 White Size L`  

| **Heuristic 1** | **Heuristic 2** |
|------------------|------------------|
| - Recommends **white shirts** of the same product type (`SHIRT`) and price (9.99).<br>- Includes diverse brands like *Wenslid* and *Miss Chievous*.<br>- Focuses on **functional similarity** (type, color, price). | - Recommends items from the **same brand** (`HX-Kingdom Fashion T-shirts`).<br>- Titles include keywords like *"Special Olympics World Games"*.<br>- Prices vary slightly (7.91–9.99) but prioritize brand and title relevance. |

**Comparison**:  
- **Heuristic 1** provides generic matches for users prioritizing product type and budget.  
- **Heuristic 2** targets brand loyalty and thematic consistency (e.g., event-specific shirts).  
- **Overlap**: Only the original query item appears in both results.  



#### **Results: Request 2**

1. **Heuristic 1 (Product-Type + Color + Price):**

In [ ]:
results["FeatherLite Ladies' Moisture Free Mesh Sport Shirt, White , XXX-Large"][0]

2. **Heuristic 2 (Brand + Title + Price)**  

In [ ]:
results["FeatherLite Ladies' Moisture Free Mesh Sport Shirt, White , XXX-Large"][1]

**Discussion of Results**

**Query**: `FeatherLite Ladies' Moisture Free Mesh Sport Shirt, White, XXX-Large`  

| **Heuristic 1** | **Heuristic 2** |
|------------------|------------------|
| - Recommends **white shirts** of the same product type (`SHIRT`) and price (20–21).<br>- Includes brands like *Clique/New Wave* and *Focal20*.<br>- Prioritizes **color and price similarity**. | - Recommends **FeatherLite brand** shirts in various colors (White, Red, Navy).<br>- Prices vary widely (20.54–27.16).<br>- Emphasizes **brand loyalty** and title keywords like "Moisture Free Mesh". |

**Comparison**:  
- **Heuristic 1** delivers functional matches but lacks brand specificity.  
- **Heuristic 2** sacrifices price consistency for brand alignment.  
- **Key Insight**: Heuristic 2’s recommendations include the same product in different sizes/colors, useful for brand-loyal users.  




#### **Results: Request 3**

1. **Heuristic 1 (Product-Type + Color + Price):**

In [ ]:
results["Supernatural Chibis Sam Dean And Castiel O Neck T-shirts For Female Purple L"][0]

2. **Heuristic 2 (Brand + Title + Price)**  

In [ ]:
results["Supernatural Chibis Sam Dean And Castiel O Neck T-shirts For Female Purple L"][1]

**Discussion of Results**

 **Query**: `Supernatural Chibis Sam Dean And Castiel O Neck T-shirts For Female Purple L`  

| **Heuristic 1** | **Heuristic 2** |
|------------------|------------------|
| - Recommends **purple shirts** of the same product type (`SHIRT`) and price (7.35–7.49).<br>- Includes brands like *Bella+Canvas* and *Soprano*.<br>- Prioritizes **color and price similarity**. | - Recommends **FNC7C brand** items with titles containing "Supernatural Chibis".<br>- Prices are consistent (7.39–7.99).<br>- Focuses on **brand and thematic keywords** (e.g., "Sam Dean And Castiel"). |

**Comparison**:  
- **Heuristic 1** provides budget-friendly options but ignores thematic relevance.  
- **Heuristic 2** aligns with the query’s theme but includes irrelevant colors (e.g., Black, SkyBlue).  



#### **Key Insights**

**Heuristic 1 (Product-Type + Color + Price)**:  
   - **Strengths**: Reliable for users prioritizing functional attributes (type, color, price).  
   - **Weaknesses**:  
     - Limited brand diversity.  
     - Struggles with thematic or keyword-based relevance.  

**Heuristic 2 (Brand + Title + Price)**:  
   - **Strengths**: Effective for brand-loyal users or keyword-driven searches (e.g., event-specific shirts).  
   - **Weaknesses**:  
     - Ignores color consistency (e.g., recommending Black shirts for a Purple query).  
     - Price ranges vary significantly.  

**Trade-offs**:  
   - Heuristic 1 sacrifices brand/keyword relevance for functional similarity.  
   - Heuristic 2 sacrifices color/price consistency for brand and thematic alignment.  

### **Study 4: Collaborative Filtering Recommendation System**

#### **Feature Engineering**

In [ ]:
# Filter out users and items with insufficient interactions
user_counts = reviews['review_userId'].value_counts()
item_counts = reviews['asin'].value_counts()
reviews = reviews[reviews['review_userId'].isin(user_counts[user_counts >= 5].index)] # Users who gave at least 5 reviews
reviews = reviews[reviews['asin'].isin(item_counts[item_counts >= 5].index)] # Items with at least 5 reviews

 #### **Create a User-Item Rating Matrix**

In [ ]:
import pandas as pd
import numpy as np


# Load dataset
reviews = pd.read_csv("Clothing-Reviews.csv")
reviews = reviews[['review_userId', 'asin', 'review_score']]

# Preprocessing: Drop missing values and filter users/items with sufficient interactions
reviews = reviews.dropna()
user_counts = reviews['review_userId'].value_counts()
item_counts = reviews['asin'].value_counts()

# Filter users with ≥5 reviews and items with ≥5 ratings
reviews = reviews[reviews['review_userId'].isin(user_counts[user_counts >= 5].index)]
reviews = reviews[reviews['asin'].isin(item_counts[item_counts >= 5].index)]

# Create Surprise Dataset
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(reviews[['review_userId', 'asin', 'review_score']], reader)

# Split data into train (90%) and test (10%)
trainset, testset = train_test_split(data, test_size=0.1, random_state=42)

# Train SVD model
model = SVD(n_factors=50, random_state=42)
model.fit(trainset)

# Predict on test set
predictions = model.test(testset)

# Evaluation 1: MSE
mse = mean_squared_error([pred.r_ui for pred in predictions], [pred.est for pred in predictions])
print(f"MSE: {mse:.3f}")

# Evaluation 2: Precision@K and MRR
def evaluate_precision_mrr(predictions, threshold=4, k=10):
    binary_preds = []
    for pred in predictions:
        actual = 1 if pred.r_ui >= threshold else 0
        estimated = 1 if pred.est >= threshold else 0
        binary_preds.append((actual, estimated, pred.uid, pred.iid))

    # Precision@K and MRR
    precision_at_k = {5: [], 10: []}
    mrr = []

    for user in set([pred[2] for pred in binary_preds]):
        user_preds = [p for p in binary_preds if p[2] == user]
        user_preds_sorted = sorted(user_preds, key=lambda x: x[1], reverse=True)

        # Precision@5 and @10
        for k_val in [5, 10]:
            top_k = user_preds_sorted[:k_val]
            relevant = sum([1 for p in top_k if p[0] == 1])
            precision = relevant / k_val if k_val > 0 else 0
            precision_at_k[k_val].append(precision)

        # MRR
        for rank, p in enumerate(user_preds_sorted, 1):
            if p[0] == 1:
                mrr.append(1 / rank)
                break

    return {
        "P@5": np.mean(precision_at_k[5]),
        "P@10": np.mean(precision_at_k[10]),
        "MRR": np.mean(mrr)
    }

metrics = evaluate_precision_mrr(predictions)
print(f"Precision@5: {metrics['P@5']:.3f}")
print(f"Precision@10: {metrics['P@10']:.3f}")
print(f"MRR: {metrics['MRR']:.3f}")

# Simulate requests for 3 users
def get_top_recommendations(user_id, model, n=10):
    user_ratings = reviews[reviews['review_userId'] == user_id]
    all_items = reviews['asin'].unique()
    rated_items = user_ratings['asin'].tolist()
    unseen_items = [item for item in all_items if item not in rated_items]

    predictions = [model.predict(user_id, item) for item in unseen_items]
    top_items = sorted(predictions, key=lambda x: x.est, reverse=True)[:n]

    return [(item.iid, item.est) for item in top_items]

# Example users
users = reviews['review_userId'].sample(3).tolist()
for user in users:
    recommendations = get_top_recommendations(user, model)
    print(f"\nTop 10 recommendations for User {user}:")
    for item, score in recommendations:
        print(f"Item {item} (Predicted Rating: {score:.2f})")